# Exercise 07 — MQTT: Publish & Subscribe
### 📡 Event-driven communication for IoT sensors

REST is great for reading or updating data on demand.
But what if you want to be **notified the moment** a sensor sends a value?
That is where **MQTT** shines.

MQTT is a **publish/subscribe** protocol:
- **Publishers** send messages to a **topic** on a **broker**
- **Subscribers** tell the broker which topics they care about
- The **broker** routes each message to all matching subscribers
- Publisher and subscriber never talk directly — they are decoupled

We will use a **free public broker** (`broker.hivemq.com`) — no installation needed.

---

In [1]:
# Install if needed:  pip install paho-mqtt
import paho.mqtt.client as mqtt
import json, time, threading

BROKER = "broker.hivemq.com"
PORT   = 1883

print(f"Will connect to public broker: {BROKER}:{PORT}")
print("No installation required — this is a free shared broker.")

Will connect to public broker: broker.hivemq.com:1883
No installation required — this is a free shared broker.


## 1️⃣  Subscribe first — then publish

The subscriber must be ready *before* the publisher sends, otherwise messages are missed.
We set up a subscriber in a background thread, then publish from the main thread.

In [2]:
# ── Use a unique prefix so you don't collide with other students ────────────
# Change STUDENT_ID to something unique (your name, student number, etc.)
STUDENT_ID = "student_abc"    # <-- change this!
BASE_TOPIC = f"smartcity/turin/{STUDENT_ID}"

print(f"Your base topic: {BASE_TOPIC}")
print(f"  Sensor topic : {BASE_TOPIC}/TRN-001/temperature")

Your base topic: smartcity/turin/student_abc
  Sensor topic : smartcity/turin/student_abc/TRN-001/temperature


In [3]:
received_messages = []   # shared list — subscriber writes, we read

# ── Subscriber ─────────────────────────────────────────────────────────────
def on_connect_sub(client, userdata, flags, rc):
    if rc == 0:
        print("📥 Subscriber connected")
        # Subscribe to ALL topics under our base
        client.subscribe(f"{BASE_TOPIC}/#")
        print(f"   Subscribed to: {BASE_TOPIC}/#")
    else:
        print(f"Connection failed, code {rc}")

def on_message(client, userdata, msg):
    payload = msg.payload.decode()
    print(f"  📨 [{msg.topic}] → {payload}")
    received_messages.append({"topic": msg.topic, "payload": payload})

sub_client = mqtt.Client(client_id=f"{STUDENT_ID}_sub")
sub_client.on_connect = on_connect_sub
sub_client.on_message = on_message
sub_client.connect(BROKER, PORT)
sub_client.loop_start()   # background thread

time.sleep(1.5)           # wait for connection
print("\nSubscriber is ready and listening.")

/var/folders/vs/5f81dpjd503fc4vv28ygh7qm0000gn/T/ipykernel_24015/2223783969.py:18: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  sub_client = mqtt.Client(client_id=f"{STUDENT_ID}_sub")


📥 Subscriber connected
   Subscribed to: smartcity/turin/student_abc/#

Subscriber is ready and listening.


## 2️⃣  Publish your first message

In [4]:
# ── Publisher ──────────────────────────────────────────────────────────────
pub_client = mqtt.Client(client_id=f"{STUDENT_ID}_pub")
pub_client.connect(BROKER, PORT)
pub_client.loop_start()
time.sleep(0.5)

# Publish a temperature reading
topic   = f"{BASE_TOPIC}/TRN-001/temperature"
payload = "22.4"

pub_client.publish(topic, payload)
print(f"📤 Published to [{topic}]: {payload}")

time.sleep(0.5)   # give the broker time to route the message
print(f"\nMessages received so far: {len(received_messages)}")

/var/folders/vs/5f81dpjd503fc4vv28ygh7qm0000gn/T/ipykernel_24015/3847284671.py:2: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  pub_client = mqtt.Client(client_id=f"{STUDENT_ID}_pub")


📤 Published to [smartcity/turin/student_abc/TRN-001/temperature]: 22.4
  📨 [smartcity/turin/student_abc/TRN-001/temperature] → 22.4

Messages received so far: 1


## 3️⃣  Publish JSON payloads — as your IoT sensors would

In [7]:
received_messages.clear()

sensors = [
    ("TRN-001", "temperature", 22.4),
    ("TRN-001", "humidity",    58.0),
    ("TRN-001", "pm25",        12.1),
    ("TRN-002", "temperature", 23.1),
    ("TRN-002", "humidity",    61.0),
    ("TRN-003", "pm25",        35.2),
]

print("Publishing sensor readings...\n")
for sensor_id, measurement, value in sensors:
    topic   = f"{BASE_TOPIC}/{sensor_id}/{measurement}"
    payload = json.dumps({"v": value, "t": time.time()})
    pub_client.publish(topic, payload, qos=1)
    print(f"  📤 [{topic}] → {payload}")
    time.sleep(0.1)

time.sleep(1)
print(f"\n✅ Sent 6 messages, received {len(received_messages)}")

Publishing sensor readings...

  📤 [smartcity/turin/student_abc/TRN-001/temperature] → {"v": 22.4, "t": 1779701138.6297228}
  📨 [smartcity/turin/student_abc/TRN-001/temperature] → {"v": 22.4, "t": 1779701138.6297228}
  📤 [smartcity/turin/student_abc/TRN-001/humidity] → {"v": 58.0, "t": 1779701138.7351098}
  📨 [smartcity/turin/student_abc/TRN-001/humidity] → {"v": 58.0, "t": 1779701138.7351098}
  📤 [smartcity/turin/student_abc/TRN-001/pm25] → {"v": 12.1, "t": 1779701138.840325}
  📨 [smartcity/turin/student_abc/TRN-001/pm25] → {"v": 12.1, "t": 1779701138.840325}
  📤 [smartcity/turin/student_abc/TRN-002/temperature] → {"v": 23.1, "t": 1779701138.945583}
  📤 [smartcity/turin/student_abc/TRN-002/humidity] → {"v": 61.0, "t": 1779701139.0501702}
  📨 [smartcity/turin/student_abc/TRN-002/temperature] → {"v": 23.1, "t": 1779701138.945583}
  📨 [smartcity/turin/student_abc/TRN-002/humidity] → {"v": 61.0, "t": 1779701139.0501702}
  📤 [smartcity/turin/student_abc/TRN-003/pm25] → {"v": 35.2, "t": 177

## 4️⃣  Quality of Service (QoS)

MQTT defines 3 levels of delivery guarantee — a key difference from REST:

| QoS | Guarantee | Use case |
|---|---|---|
| 0 | At most once (fire & forget) | Non-critical telemetry |
| 1 | At least once (may duplicate) | Sensor readings |
| 2 | Exactly once (4-step handshake) | Critical commands |


In [ ]:
received_messages.clear()

# Publish the same message at different QoS levels
for qos in [0, 1, 2]:
    topic   = f"{BASE_TOPIC}/TRN-001/temperature"
    payload = json.dumps({"v": 22.4, "qos_used": qos})
    result  = pub_client.publish(topic, payload, qos=qos)
    print(f"  QoS {qos} publish → rc={result.rc} (0=success)")
    time.sleep(0.3)

time.sleep(1)
print(f"\nReceived {len(received_messages)} messages")
for m in received_messages:
    print(f"  {json.loads(m['payload'])}")

## 5️⃣  Build an alert subscriber

Let's create a subscriber that listens only to PM2.5 topics and triggers an alert when the value exceeds the threshold.

In [ ]:
PM25_THRESHOLD = 20.0
alerts = []

def on_pm25_message(client, userdata, msg):
    try:
        data  = json.loads(msg.payload.decode())
        value = data.get("v", 0)
        # Extract sensor ID from topic: .../TRN-XXX/pm25
        parts     = msg.topic.split("/")
        sensor_id = parts[-2]
        if value > PM25_THRESHOLD:
            alert = f"⚠️  ALERT  {sensor_id}  PM2.5={value} µg/m³  (>{PM25_THRESHOLD})"
            print(alert)
            alerts.append(alert)
        else:
            print(f"  ✅ {sensor_id}  PM2.5={value} µg/m³  OK")
    except Exception as e:
        print(f"  Parse error: {e}")

# New subscriber only for pm25
alert_client = mqtt.Client(client_id=f"{STUDENT_ID}_alert")
alert_client.on_message = on_pm25_message
alert_client.connect(BROKER, PORT)
alert_client.subscribe(f"{BASE_TOPIC}/+/pm25")   # + wildcard for sensor ID
alert_client.loop_start()
time.sleep(0.5)

# Publish pm25 values for all 5 sensors
pm25_values = [("TRN-001", 12.1), ("TRN-002", 18.4), ("TRN-003", 35.2), ("TRN-004", 22.7), ("TRN-005", 8.3)]
print("Publishing PM2.5 readings...\n")
for sensor_id, value in pm25_values:
    topic = f"{BASE_TOPIC}/{sensor_id}/pm25"
    pub_client.publish(topic, json.dumps({"v": value, "t": time.time()}), qos=1)
    time.sleep(0.2)

time.sleep(1.5)
print(f"\nTotal alerts triggered: {len(alerts)}")

---
## 🏆 Challenges

1. Add a `retain=True` flag to one `publish()` call. Then create a new subscriber *after* publishing and observe it receives the last retained value immediately.
2. Create a subscriber on `{BASE_TOPIC}/+/temperature` that computes a running average of temperature across all sensors.
3. Create a "command" topic `{BASE_TOPIC}/commands` and publish a JSON command `{"action": "reset", "target": "TRN-001"}`. Make a subscriber that reacts to it.
4. What happens if you publish but there is no subscriber? Is the message lost? Try it and explain the difference from REST.

In [ ]:
# Your code here
